# 02 — Text Chunking: So Sánh Chiến Lược

**Mục tiêu:** Thực nghiệm và so sánh ≥ 2 chiến lược chunking (`FIXED_SIZE`, `RECURSIVE`) trên cùng một tài liệu để hiểu tác động của từng chiến lược đến số lượng và kích thước chunk.

**Cấu trúc cell theo design.md §2.5:**
1. Import & setup
2. Tải tài liệu mẫu
3. Thực nghiệm Fixed-Size Chunking
4. Thực nghiệm Recursive Chunking
5. Thực nghiệm Semantic Chunking
6. **So sánh kết quả** ← core của task S2-DE-05
7. Log thực nghiệm (placeholder cho Sprint 4)

**Phụ thuộc:** S2-DE-01 (`chunk_by_fixed_size`), S2-DE-02 (`chunk_by_recursive`)

In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / 'src').is_dir():
            return parent
    return start

PROJECT_ROOT = _find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Cell 1: Import và setup
import pandas as pd
from src.data.loader import DocumentLoader
from src.data.chunker import TextChunker
from src.models import ChunkStrategy
# from src.pipeline.experiment_tracker import ExperimentTracker

# tracker = ExperimentTracker()  # uncomment khi S4-PE-01 hoàn thành
print("✓ Import thành công")

In [ ]:
# Cell 2: Tải tài liệu mẫu
loader = DocumentLoader()
sample_path = PROJECT_ROOT / "data" / "raw" / "sample_pdf.pdf"
doc = loader.load(str(sample_path))
print(f"Nội dung: {len(doc.content)} ký tự")

In [ ]:
# Cell 3: Thực nghiệm Fixed-Size Chunking
chunker_fixed = TextChunker(strategy=ChunkStrategy.FIXED_SIZE, chunk_size=256)
chunks_fixed = chunker_fixed.chunk(doc)
print(f"Fixed-size: {len(chunks_fixed)} chunks")

In [ ]:
# Cell 4: Thực nghiệm Recursive Chunking
chunker_recursive = TextChunker(strategy=ChunkStrategy.RECURSIVE, chunk_size=256)
chunks_recursive = chunker_recursive.chunk(doc)
print(f"Recursive: {len(chunks_recursive)} chunks")

In [ ]:
# Cell 5: Thực nghiệm Semantic Chunking
chunker_semantic = TextChunker(strategy=ChunkStrategy.SEMANTIC, chunk_size=256)
chunks_semantic = chunker_semantic.chunk(doc)
print(f"Semantic: {len(chunks_semantic)} chunks")

In [ ]:
# Cell 6: So sánh kết quả — theo design.md §2.5
def compare_strategies(chunks_a, name_a, chunks_b, name_b):
    """So sánh hai chiến lược chunking.

    Returns:
      dict: {name_a: {count, avg_len, min_len, max_len}, name_b: {...}}
    """
    def _stats(chunks):
        lengths = [len(c.content) for c in chunks]
        if not lengths:  # guard: tránh ZeroDivisionError / min/max trên list rỗng
            return {"count": 0, "avg_len": 0.0, "min_len": 0, "max_len": 0}
        return {
            "count":   len(chunks),
            "avg_len": round(sum(lengths) / len(lengths), 1),
            "min_len": min(lengths),
            "max_len": max(lengths),
        }

    return {name_a: _stats(chunks_a), name_b: _stats(chunks_b)}


# So sánh Fixed-Size vs Recursive (≥ 2 chiến lược theo Yêu cầu 9.5)
stats = compare_strategies(chunks_fixed, "fixed_size", chunks_recursive, "recursive")

df = pd.DataFrame(stats).T
df.index.name = "strategy"
print("\n=== So sánh chiến lược chunking ===")
print(df.to_string())
df

In [ ]:
# Cell 7: Log thực nghiệm — placeholder cho Sprint 4 (S4-PE-01)
# ExperimentTracker chưa implement — uncomment khi S4-PE-01 hoàn thành.
# Uncommenting now sẽ crash notebook (vi phạm Yêu cầu 9.2).
#
# tracker.log_indexing(
#     doc_id=doc.doc_id,
#     chunk_strategy="fixed_size",
#     chunk_size=256,
#     num_chunks=len(chunks_fixed),
#     latency_ms=0.0,
# )
# tracker.log_indexing(
#     doc_id=doc.doc_id,
#     chunk_strategy="recursive",
#     chunk_size=256,
#     num_chunks=len(chunks_recursive),
#     latency_ms=0.0,
# )
print("(Tracker placeholder — sẽ activate ở Sprint 4 / S4-PE-01)")